# Stroop och flankertest, en jämförelse.

Vi har valt att jämföra data från Stroop och Flanker utifrån två huvudsakliga frågeställningar: 1. Vilken effekt är störst: Stroop eller Flanker. 2. Finns det en korrelation mellan Stroop effekten och Flanker effekten hos individer? 

Länk till vår repo: https://github.com/gusuny/data_analysis_project

### Importera verktyg

In [ ]:
#importerar in verktyg och bibliotek
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

## Flanker
läser in och konverterar vår excelfil

In [ ]:
#läser in vår fil
df_flanker = pd.read_excel("Raw_Flanker.xlsx")

#spara som csv i samma directory
df_flanker.to_csv("Raw_Flanker.csv", index=False)

### Utforska och få en överblick över datan
Titta efter NaN, se vilka kolumner vi behöver och hur vi på andra sätt kan förenkla vår data till vårt syfte

In [ ]:
#utforska datan

df_flanker = pd.read_csv("Raw_Flanker.csv")

print("Flanker info")
df_flanker.info()
display(df_flanker.head())


In [ ]:
#leta efter NaN

print("Tot NaN for Flanker-test")
df_flanker.isna().sum()

### Cleana vår data
Välja kolumner som vi vill ha kvar, sortera, skapa ny kolumn 'session', ta bort practiceruns, göra 'date' tydligare etc. Vi har valt att avgränsa oss genom att enbart använda data från online sessionerna (session 1), då det innehåller ett stort antal deltagare som alla genomfört uppgifterna under samma förhållanden. 

In [ ]:
# gör datumet mer lättläst
df_flanker['date'] = pd.to_datetime(df_flanker['date'].astype(str).str.zfill(6), format='%m%d%y', errors='coerce').dt.strftime('%Y-%m-%d')

In [ ]:
# sortera kronologiskt efter deltagare, datum och tid så sessionerna hamnar i rätt ordning
df_flanker = df_flanker.sort_values(by=['subjectid', 'date', 'time'])

In [ ]:
# skapa 'session'-kolumnen per deltagare (1 = tidigaste tillfället, 2 = nästa, 3 = sista)
df_flanker['session'] = df_flanker.groupby('subjectid')['date'].transform(lambda x: pd.factorize(x)[0] + 1)

In [ ]:
# ta bort övningsförsök
df_clean = df_flanker[df_flanker['values.practice'] == 0].copy()

In [ ]:
#kontrollera hur det ser ut för en person med 1 eller flera sessions
print(df_clean[df_clean['subjectid'] == 484][['subjectid', 'date', 'time', 'session']].drop_duplicates())

In [ ]:
# ta bort session 2 och 3 och kontrollera att de tagits bort
df_clean = df_clean[~df_clean['session'].isin([2, 3])].copy()
print(df_clean['session'].value_counts())

In [ ]:
# välj enbart de kolumner vi vill behålla (inklusive 'session')
kolumner_att_behålla = ['date', 'time', 'subjectid', 'session', 'correct', 'values.congruence', 'latency']

In [ ]:
# kontrollera att kolumnerna finns i df_clean och kontrollera att alla kolumner finns i head
df_clean = df_clean[[col for col in kolumner_att_behålla if col in df_clean.columns]]
df_clean.head()

### Ta bort outliers

In [ ]:
#antal unika deltagare
antal_unika = df_clean['subjectid'].nunique()
print(f"Antal unika deltagare: {antal_unika}")

In [ ]:
# Beräkna genomsnittlig mängd rätt per test och betingelse
df_clean.groupby('values.congruence')['correct'].mean()

I cellen nedan har vi valt att ta bort deltagare med mindre än 80 % rätt. Vi motiverar detta med att testen bygger på att deltagaren har förstått instruktionen och försöker svara rätt: En deltagare med många fel kan indikera att denna har missförstått uppgiften, gissat eller tappat motivationen. Deras reaktionstider mäter då inte förmågan att hantera kognitiv konflikt. 

I Flanker finns två svarsalternativ så gissningsnivån ligger på 50 %. Test av genomsnittliog träffsäkerhet per test och betingelse i detta dataset ligger i sin tur på 97-99 % (se cellen ovan). En gräns på 80 % kräver att deltagare presterar bättre än slumpen, men är inte så sträng att den inte tillåter vanliga misstag. 

In [ ]:
# exkludera personer med låg andel rätt (under 80% rätt)
ACC_GRANS = 0.80  # 80 %

#skapa var acc_flanker genom att gruppera df_clean med 'subjectid' och medelvärdet av 'correct' 
acc_flanker = df_clean.groupby("subjectid")["correct"].mean()

#skapar var 'låg', plockar ut alla subjectids som ligger under 80% och sparar dem i var. 'låg'
låg = acc_flanker[acc_flanker < ACC_GRANS].index  # deltagare UNDER 80 % i Flanker

#printar ut deltagare före och efter filtrering
print(f"Deltagare före: {df_clean['subjectid'].nunique()}")
print(f"Exkluderas ({len(låg)} st):", list(låg))

#behåller personer som inte finns i var. 'låg' (behåller deltagare med ≥ 80 %)
df_clean = df_clean[~df_clean["subjectid"].isin(låg)].copy() 


#skriver ut deltagare efter
print(f"Deltagare efter: {df_clean['subjectid'].nunique()}")

I cellen nedan har vi valt att ta bort trials under 200 ms och över 3000 ms. Svar under 200 ms betraktas som förhastade svar, medan svar över 3000 ms betraktas som uttryck för distraktion eller ouppmärksamhet.

In [ ]:
# exkludera trials under 200 ms eller över 3000 ms
#spara antalet rader innan vi tar bort gränsvärden
antal_fore = len(df_clean)

#sätta gränsvärden för outliers (200ms nedre gräns, 3000ms övre gräns)
df_clean = df_clean[(df_clean['latency'] >= 200) & (df_clean['latency'] <= 3000)].copy()

#ta reda på hur många vi tagit bort genom att jämföra före och efter
antal_efter = len(df_clean)
antal_borttagna = antal_fore - antal_efter

print(f"Antal borttagna försök: {antal_borttagna}")

### Kontrollera våran data
Se över att allt ser korrekt ut och som vi vill ha det, kan vi göra ytterligare förbättringar.

In [ ]:
#kontrollera våran data efter vi cleanat den

display(df_clean.head())
df_clean.info()

#tittar hur många unika subjectids(deltagare)
df_clean['subjectid'].nunique()

antal_unika = df_clean['subjectid'].nunique()
print(f"Antal unika deltagare: {antal_unika}")

In [ ]:
#spara ner allt till EN gemensam CSV-fil (ta bort # för att spara ner en ny csv fil med cleanad data)

#df_clean.to_csv("flanker_all_sessions_combined.csv", index=False)

print("Ta bort # ovan för att spara ner en ny 'cleanad' csv, 'flanker_all_sessions_combined.csv'.")

# Stroop

läs in fil, konvertera till csv

In [ ]:
#Läser in ecxel filen

df_stroop = pd.read_excel("Raw_Stroop.xlsx")

#Spara som csv i samma directory

df_stroop.to_csv("Raw_Stroop.csv", index=False)

#Läs in csv filen
df_stroop = pd.read_csv("Raw_Stroop.csv")

### Utforska och få en överblick över datan
titta efter NaN, se vilka kolummner vi behöver och hur vi på andra sätt kan förenkla vår data för vårt syfte

In [ ]:
#explore the data

print("Stroop info")
df_stroop.info()
display(df_stroop.head())

In [ ]:
#kolla efter NaN

print("Tot NaN for Stroop-test")
df_stroop.isna().sum()

### Cleana vår data

In [ ]:
# gör datumet mer lättläst
df_stroop['date'] = pd.to_datetime(df_stroop['date'].astype(str).str.zfill(6), format='%m%d%y', errors='coerce').dt.strftime('%Y-%m-%d')

In [ ]:
# sortera kronologiskt efter deltagare, datum och tid så sessionerna hamnar i rätt ordning
df_stroop = df_stroop.sort_values(by=['subjectid', 'date', 'time'])

In [ ]:
# skapa 'session'-kolumnen per deltagare (1 = tidigaste tillfället, 2 = nästa, 3 = sista)
df_stroop['session'] = df_stroop.groupby('subjectid')['date'].transform(lambda x: pd.factorize(x)[0] + 1)

In [ ]:
# välj enbart de kolumner vi vill behålla (inklusive 'session')
kolumner_att_behålla = ['date', 'time', 'subjectid', 'session', 'correct', 'latency', 'values.congruency']

In [ ]:
# kontrollera att kolumnerna finns
df_clean_stroop = df_stroop[kolumner_att_behålla].copy()

In [ ]:
#kontrollera hur det ser ut för en person med 1 eller flera sessions
print(df_clean_stroop[df_clean_stroop['subjectid'] == 484][['subjectid', 'date', 'time', 'session']].drop_duplicates())

In [ ]:
# ta bort session 2 och 3
df_clean_stroop = df_clean_stroop[~df_clean_stroop['session'].isin([2, 3])].copy()

In [ ]:
df_clean_stroop.head()

### Ta bort outliers

In [ ]:
#antal unika deltagare
antal_unika = df_clean_stroop['subjectid'].nunique()
print(f"Antal unika deltagare: {antal_unika}")

In [ ]:
# beräkna genomsnittlig träffsäkerhet per test och betingelse för Stroop
df_clean_stroop.groupby('values.congruency')['correct'].mean()

I cellen nedan har vi likt för Flanker valt att ta bort deltagare med mindre än 80 % rätt. Vi motiverar detta med att testen bygger på att deltagaren har förstått instruktionen och försöker svara rätt: En deltagare med många fel kan indikera att denna har missförstått uppgiften, gissat eller tappat motivationen. Deras reaktionstider mäter då inte förmågan att hantera kognitiv konflikt. 

I Stroop finns fyra svarsalternativ så gissningsnivån ligger på 25 %. Test av genomsnittlig träffsäkerhet per test och betingelse i detta dataset ligger i sin tur på 94-97 % (se cellen ovan). Vi utgår från samma gräns som för Flanker (80 %), vilket ligger över slumpnivå men tillåter samtidigt för misstag. 

In [ ]:
#exkludera personer med låg andel rätt
ACC_GRANS = 0.80  # 80 %

acc_stroop = df_clean_stroop.groupby("subjectid")["correct"].mean()

låg = acc_stroop[acc_stroop < ACC_GRANS].index  # deltagare UNDER 80 % i Stroop

print(f"Deltagare före: {df_clean_stroop['subjectid'].nunique()}")
print(f"Exkluderas ({len(låg)} st):", list(låg))

df_clean_stroop = df_clean_stroop[~df_clean_stroop["subjectid"].isin(låg)].copy()  # behåller deltagare med ≥ 80 %

print(f"Deltagare efter: {df_clean_stroop['subjectid'].nunique()}")

I cellen nedan har vi valt att ta bort trials under 200 ms och över 3000 ms. Svar under 200 ms betraktas som förhastade svar, medan svar över 3000 ms betraktas som uttryck för distratkion eller ouppmärksamhet. 

In [ ]:
# exkludera trials under 200 ms eller över 3000 ms

#spara antalet rader innan vi tar bort gränsvärden
antal_fore = len(df_clean_stroop)

#sätta gränsvärden för outliers (200ms nedre gräns, 3000ms övre gräns)
df_clean_stroop = df_clean_stroop[(df_clean_stroop['latency'] >= 200) & (df_clean_stroop['latency'] <= 3000)].copy()

#ta reda på hur många vi tagit bort genom att jämföra före och efter
antal_efter = len(df_clean_stroop)
antal_borttagna = antal_fore - antal_efter

print(f"Antal borttagna försök: {antal_borttagna}")

### Kontrollera vår data
Se över att allt ser korrekt ut och som vi vill ha det, kan vi göra ytterligare förbättringar

In [ ]:
#kontrollera våran data efter vi cleanat den

display(df_clean_stroop.head())
df_clean_stroop.info()

#titta efter unika deltagare
df_clean_stroop['subjectid'].nunique()

antal_unika = df_clean_stroop['subjectid'].nunique()
print(f"Antal unika deltagare: {antal_unika}")

### Aggregera och merge stroop och flanker till df_final

In [ ]:
#aggregera data för stroop

stroop_agg = df_clean_stroop[df_clean_stroop['correct'] == 1].groupby(['subjectid', 'session', 'values.congruency'])['latency'].mean().reset_index() #filtrera på korrekta svar och aggregera medelvärde av reaktionstid per deltagare, session och congruence
stroop_agg.rename(columns={'latency': 'stroop_latency', 'values.congruency':'congruence'}, inplace=True) #döp om kolumnerna för att matcha efterföljande flanker_agg

#aggregera data för flanker
flanker_agg = df_clean[df_clean['correct'] == 1].groupby(['subjectid', 'session', 'values.congruence'])['latency'].mean().reset_index() #filtrera på korrekta svar och aggregera medelvärde av reaktionstid per deltagare, session och congruence
flanker_agg.rename(columns={'latency': 'flanker_latency', 'values.congruence': 'congruence'}, inplace=True) # döp om kolumner

# Merga på kolumnerna id, session och congruence
df_final = pd.merge(stroop_agg, flanker_agg, on=['subjectid', 'session', 'congruence'])

In [ ]:
#kontrollera df_final

df_final.head()

### Visualisering av resultaten

In [ ]:
#Långt format: en rad per deltagare × test × kongruens, används till violinplotten och mediantabellen.

df_long = df_final.melt(
    id_vars=['subjectid', 'session', 'congruence'],
    value_vars=['stroop_latency', 'flanker_latency'],
    var_name='Test',
    value_name='Latency (ms)'
)
df_long['Test'] = df_long['Test'].map({'stroop_latency': 'Stroop', 'flanker_latency': 'Flanker'})
df_long['congruence'] = df_long['congruence'].map({1: 'Kongruent', 2: 'Inkongruent'})

# interferenseffekt, en rad per deltagare × test, används som grund för spridningsdiagrammet.
df_effekt = (df_long
             .pivot_table(index=['subjectid', 'session', 'Test'],
                          columns='congruence',
                          values='Latency (ms)')
             .reset_index()
             .rename_axis(columns=None))
df_effekt['Interferenseffekt (ms)'] = df_effekt['Inkongruent'] - df_effekt['Kongruent']

# en rad per deltagare, Stroop och Flanker som kolumner, används till spridningsdiagrammet.
df_scatter = (df_effekt
              .pivot(index=['subjectid', 'session'],
                     columns='Test',
                     values='Interferenseffekt (ms)')
              .reset_index()
              .rename_axis(columns=None)
              .dropna())

print(f"Antal deltagare: {df_scatter['subjectid'].nunique()}")

## Skiljer sig reaktionstid mellan stroop och flanker test?:

In [ ]:
# Violin plot
# Fördelningen av medelreaktionstider i varje test och betingelse. 

plt.figure(figsize=(9, 6))
sns.violinplot(data=df_long, x='Test', y='Latency (ms)', hue='congruence',
              hue_order=['Kongruent', 'Inkongruent'],
               split=True, inner='quartile', palette='Set2')
plt.title('Jämförelse av reaktionstid: Flanker vs Stroop (delat på kongruens)')
plt.xlabel('')
plt.ylabel('Reaktionstid (ms)')
plt.legend(title='Kongruens')
plt.tight_layout()
plt.show()

# medianer per test och kongruens
medianer = df_long.groupby(['Test', 'congruence'])['Latency (ms)'].median()

tabell = (medianer.unstack()
          [['Kongruent', 'Inkongruent']]            
          .reindex(['Stroop', 'Flanker'])           
          .round(0)
          .astype(int))
tabell.index.name = None
tabell.columns.name = None

print("Median reaktionstid (ms), korrekta svar")
print(tabell)

**Tolkning:**

Figuren visar fördelningen av deltagarnas medelreaktionstider för korrekta trials, uppdelat på test och kongruens. De streckade linjerna markerar medianerna. 

Reaktionstiderna var generellt längre i Stroop: Medianen för kongruenta trials låg på 726 ms och 836 ms för inkongruenta trials, medan medianen för kongruenta trials för flanker låg på 413 ms och 456 ms för inkongruenta.  Medianen ökar därmed med 110 ms hos Stroop i jämförelse med ca 40 ms hos flanker. Interferenseffekten verkar därmed vara större hos Stroop än för flanker. Stroop violinen uppvisar även större skillnader mellan deltagare: RT för Stroop sträcker sig mellan ca 400 ms till nästan 2000 ms, medan flanker ryms inom ca 300 till 750 ms. 

Samtidigt ligger den orangea (inkongruenta) halvan högre än den gröna (kongruenta) i båda fallen, vilket synnligör hur reaktionstiden var längre för imkongruenta trials för båda testen. Båda figurerna har även svansar uppåt, vilka synnligör några få långsamma deltagare. 

## Finns det ett samband mellan effekterna?:
### Har personer med stor stroop effekt också stor flanker effekt?


In [ ]:
# Spridningsdiagram
#Finns det ett samband mellan en persons Stroop effekt och Flankereffekt? 
r = df_scatter['Stroop'].corr(df_scatter['Flanker']) # räknar ut pearson korrelation för båda effekterna.

sns.regplot(data=df_scatter, x='Flanker', y='Stroop', scatter_kws={'alpha': 0.4}) # stroop effekt på y-axeln och flanker på x-axeln.
plt.title(f'Stroop- vs flankereffekt (r = {r:.2f})')
plt.xlabel('Flankereffekt (ms)'); plt.ylabel('Stroopeffekt (ms)')
plt.show()

**Tolkning**

Figuren visar ett spridningsdiagram där varje punkt är en deltagare. X-axeln visar personens Flanker effekt, medan y-axeln visar personens Stroopeffekt. 

Spridningsdiagrammet visar att sambandet mellan inteferenseffekterna hos Stroop och flanker är svagt, r=0.10. Punktmolnet är nästintill runt och linjen lutar bara svagt. 

## Statistisk analys


In [ ]:
#skapa en csv fil som vi kan köra in JASP

# pivotera datan och behåll full kontroll över kolumnordningen
df_anova = df_final.pivot_table(
    index='subjectid', 
    columns='congruence', 
    values=['stroop_latency', 'flanker_latency']
)

# ordna om kolumnerna så att vi vet exakt var de hamnar:
# (congruence, 1 = kongruent, 2 = inkongruent)
df_anova = df_anova[[
    ('stroop_latency', 1), 
    ('stroop_latency', 2), 
    ('flanker_latency', 1), 
    ('flanker_latency', 2)
]]

# ge kolumnerna enkla och tydliga namn
df_anova.columns = [
    'stroop_cong', 
    'stroop_incong', 
    'flanker_cong', 
    'flanker_incong'
]

df_anova = df_anova.reset_index().dropna()

# 4. Spara till CSV för JASP
df_anova.to_csv("df_anova.csv", index=False)


In [ ]:
#lägga in skärmdump på Anovan från JASP

from IPython.display import Image

Image(filename='anovapic2.png', width=500)

In [ ]:
# lägg in skärmdump från t-test

from IPython.display import Image

Image(filename='t_test(effekt).png', width=500)

**Tolkning**
En 2x2 repeated measure ANOVA genomfördes på medelreaktionstider för korrekta svar. De oberoende variablerna var typ av test (Stroop, flanker) och kongruens (kongruent, inkongruent). 

Det fanns en huvudeffekt av test, F(1, 458) = 2240, p<0.001, η²ₚ = 0.83, där reaktionstiderna var längre i Stroop än i flanker. Det fanns också en huvudeffekt av kongruens, F(1.458) = 1238, p< 0.001, partial η² = 0.73, där reaktionstiderna var längre för inkongruenta än för kongruenta trials sett över båda testen. 

Interaktionen mellan test och kongruens var signifikant, F(1, 458) = 242.5, p< 0.001, η²ₚ  = 0.35, vilket visar att effekten av kongruens skiljer sig mellan testen. Den deskriptiva statistiken visar riktningen på denna effekt: Inteferenseffekten (skillnaden i medel RT mellan kongruent och inkongruent) var större i Stroop (111,8 ms) än i flanker (45.6). 

Ett uppföljande parat t-test bekräftade att interferenseffekten var större i Stroop (M=111.9 ms, SD = 90.0) än i flanker (M=45.6 ms,  SD = 25.5), t(458) = 15.57, p<0.001, d = 0.73, 95 % KI [0.62; 0.83]. Resultatet bekräftades av ett Wilcoxons teckenrangetest, z = 13.64, p>0.001, r = 0.73

In [ ]:
# Pearson korrelation: stroop effekt och flanker effekt

from scipy.stats import pearsonr

r, p = pearsonr(df_scatter['Stroop'], df_scatter['Flanker'])

print(f"Pearson r: {r:.2f}")
print(f"p-värde: {p:.4f}")

**Tolkning**
Korrelationen mellan Stroop och Flankereffekten var statistiskt signifikant men svagt, r=0.10, p= 0.039. Flankereffekten förklarade endast ca 1 % av variationen i Stroopeffekten, vilket tyder på att en persons inteferenseffekt i det ena testet säger mycket lite om effekten i det andra. Detta kan möjligtvis bero på att testen fångar olika kognitiva processer: semantisk/visuell konflikt i Stroop och spatial/visuell konflikt i flanker. 

## AI Disclosure
Generativ AI har använts som stöd för att ge förslag på och förbättra python kod. Ideerna bakom aanalysprocessen och för behandlingen av datan är gruppens egna. Samtliga gruppens medlemmar tar ansvar för allt innehåll i notebooken.
